In [3]:
import pandas as pd
import numpy as np
import ast
import joblib
from geopy.distance import geodesic
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score, accuracy_score

# 1. Load Data (Ensuring names match your screenshot)
try:
    vol_df = pd.read_csv('synthetic_volunteers_demo.csv')
    task_df = pd.read_csv('synthetic_tasks_demo.csv')
except FileNotFoundError:
    print("Files not found! Make sure you ran the generator script first.")

# Convert string representation of list back to actual list
vol_df['skill_set'] = vol_df['skill_set'].apply(ast.literal_eval)

# 2. Advanced Feature Engineering
def get_match_features(volunteer, task):
    # Geographic Distance
    dist = geodesic((volunteer['lat'], volunteer['lon']), (task['lat'], task['lon'])).km
    
    # Skill Match (binary)
    skill_match = 1 if task['required_skill'] in volunteer['skill_set'] else 0
    
    # Time Logic (Medical/Emergency = always match, Others = 9-5)
    task_time = pd.to_datetime(task['timestamp'])
    is_emergency = task['urgency'] >= 4
    is_working_hour = 9 <= task_time.hour <= 17
    
    time_score = 0
    avail = volunteer['availability']
    if avail == 'Full-time' or is_emergency:
        time_score = 1
    elif avail == 'Weekday' and is_working_hour:
        time_score = 1
    elif avail == 'Evening' and task_time.hour > 17:
        time_score = 1
        
    return [dist, skill_match, time_score, task['urgency']]

# 3. Generating Pairwise Training Data
import random
import pandas as pd
from sklearn.model_selection import train_test_split

# 3. Generating Pairwise Training Data (WITH NOISE)
import random
import pandas as pd
from sklearn.model_selection import train_test_split

# 3. Generating a BALANCED Pairwise Training Data (WITH 5% NOISE)
X_list, y_list = [], []
NOISE_LEVEL = 0.05  

pos_count = 0
neg_count = 0
TARGET_SAMPLES = 2000 # 2000 good, 2000 bad

print("Generating balanced dataset... this might take a few seconds.")

# Keep randomly pairing until we have exactly 2000 of both classes
while pos_count < TARGET_SAMPLES or neg_count < TARGET_SAMPLES:
    v = vol_df.sample(1).iloc[0]
    t = task_df.sample(1).iloc[0]
    feats = get_match_features(v, t)
    
    # Standard Logic
    ideal_label = 1 if (feats[1] == 1 and feats[0] < 8 and feats[2] == 1) else 0
    
    # Add to list if we still need samples for this specific class
    if ideal_label == 1 and pos_count < TARGET_SAMPLES:
        final_label = 0 if random.random() < NOISE_LEVEL else 1 # Apply Noise
        X_list.append(feats)
        y_list.append(final_label)
        pos_count += 1
        
    elif ideal_label == 0 and neg_count < TARGET_SAMPLES:
        final_label = 1 if random.random() < NOISE_LEVEL else 0 # Apply Noise
        X_list.append(feats)
        y_list.append(final_label)
        neg_count += 1

X = pd.DataFrame(X_list, columns=['distance', 'skill_match', 'time_match', 'urgency'])
y = pd.Series(y_list)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Generated {len(X)} perfectly balanced samples with a {NOISE_LEVEL*100}% noise factor.")
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Generated {len(X)} training samples with a {NOISE_LEVEL*100}% noise factor.")
# 4. Model Training & Comparison with Hyperparameters
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=100,      # Number of trees
        max_depth=5,           # Limits how deep the tree goes to prevent pure memorization
        random_state=42        # Ensures the exact same results every time you run it
    ),
    "XGBoost": XGBClassifier(
        eval_metric='logloss', # Fixed the warning you had earlier!
        learning_rate=0.1,     # How fast the model adapts
        max_depth=5, 
        random_state=42
    ),
    "LightGBM": LGBMClassifier(
        verbose=-1, 
        learning_rate=0.1, 
        max_depth=5, 
        random_state=42
    )
}

print("--- Model Comparison ---")
for name, model in models.items():
    model.fit(X_train, y_train)
    p = model.predict(X_test)
    print(f"{name} -> Accuracy: {accuracy_score(y_test, p):.4f}, F1: {f1_score(y_test, p):.4f}")

# Save the XGBoost model as the default
import joblib
joblib.dump(models["XGBoost"], 'volunteer_matcher.pkl')
print("\nModel saved successfully as 'volunteer_matcher.pkl'")

Generating balanced dataset... this might take a few seconds.
Generated 4000 perfectly balanced samples with a 5.0% noise factor.
Generated 4000 training samples with a 5.0% noise factor.
--- Model Comparison ---
Random Forest -> Accuracy: 0.9625, F1: 0.9610
XGBoost -> Accuracy: 0.9625, F1: 0.9610
LightGBM -> Accuracy: 0.9625, F1: 0.9610

Model saved successfully as 'volunteer_matcher.pkl'


In [4]:
import pandas as pd
import folium
from folium.plugins import HeatMap

# 1. Load the NGO Tasks Data
tasks_df = pd.read_csv('synthetic_tasks_demo.csv') # Change name if you used the '_demo' version

# 2. Initialize the Map (Centered on Delhi)
# Coordinates for New Delhi
map_center = [28.6139, 77.2090]
urgency_map = folium.Map(location=map_center, zoom_start=11, tiles='CartoDB Positron')

# 3. Format Data for the Heatmap
# Folium HeatMap needs a list of [Latitude, Longitude, Weight]
# We use your 'urgency' column as the weight!
heat_data = []
for index, row in tasks_df.iterrows():
    heat_data.append([row['lat'], row['lon'], row['urgency']])

# 4. Add the Heatmap layer
HeatMap(
    heat_data, 
    radius=15,      # Size of the "heat" blobs
    blur=10,        # Smoothness
    max_val=5       # Since your max urgency is 5
).add_to(urgency_map)

# 5. Save and Display
urgency_map.save('community_needs_map.html')
print("Map saved as 'community_needs_map.html'. Open it in your browser!")

# If you run this in Jupyter, the map will appear right below the cell:
urgency_map

Map saved as 'community_needs_map.html'. Open it in your browser!


C:\Users\Gatik Singh\AppData\Local\Temp\ipykernel_20068\2452231000.py:21: UserWarning: The `max_val` parameter is no longer necessary. The largest intensity is calculated automatically.
  HeatMap(
